# `edge_eeg` Colab 全量三轮训练与论文第 3、4、5 章实验一键跑通
一次性跑完硕士论文第 3、4、5 章所需的最终指标、消融实验与端侧 C99 INT8 权重头文件 (`model_weights.h`)：
- **第一轮（第 3 章 Stage 1 全量 VAD）**：训练 8 通道 `MicroVADNet`（INT8 QAT），刷出 **85%+ F1-macro**。
- **第二轮（第 4 章 Stage 2 探顶 Teacher）**：训练 306 通道 `TeacherConformerTCN`，探明高密度 MEG 理论上限并自动保存 `teacher_logits.pt`。
- **第三轮（第 4/5 章 Stage 2 端侧 Student + 蒸馏 + INT8 QAT）**：加载 `teacher_logits.pt` 蒸馏训练 8 通道 `Rep-TCN-EEGNet`，验证结构重参数化 SRAM 削减与百分位截断 INT8 QAT，导出最终 `model_weights.h`。

In [ ]:
# === Cell 1: 挂载 Drive、拉取代码与建立 /content/studies_links 软链桥 ===
from google.colab import drive
import os
import shutil
import subprocess
from pathlib import Path

drive.mount("/content/drive")
DRIVE_WS = Path("/content/drive/MyDrive/EdgeEEG_Workspace")
for sub in ("data", "archives", "runs"):
    (DRIVE_WS / sub).mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/wdongdong0708-jpg/EmbeddedBrainDecoding.git"
CODE_DIR = Path("/content/code")
if CODE_DIR.exists() and not (CODE_DIR / ".git").exists():
    shutil.rmtree(CODE_DIR)
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
assert (CODE_DIR / "scripts" / "colab_setup.py").exists(), "Repository clone failed!"

LINKS_DIR = Path("/content/studies_links")
LINKS_DIR.mkdir(parents=True, exist_ok=True)
for ds_name in ("libribrain", "auditory_eeg"):
    target = DRIVE_WS / "data" / ds_name
    target.mkdir(parents=True, exist_ok=True)
    link_path = LINKS_DIR / ds_name
    if link_path.is_symlink() or link_path.exists():
        link_path.unlink()
    os.symlink(target, link_path)
print("Code ready at:", CODE_DIR)
print("Symlink bridge ready:", list(LINKS_DIR.iterdir()))

In [ ]:
# === Cell 2: 初始化兼容 Python 3.13+ 的安全虚拟环境 ===
!python3 /content/code/scripts/colab_setup.py --venv /content/eeg-venv --repo /content/code
PYTHON = "/content/eeg-venv/bin/python"

In [ ]:
# === Cell 3: 智能解包或构建本地 NVMe 特征缓存 (/content/local_cache) ===
import os
from pathlib import Path

os.environ["EDGEEEG_STUDIES"] = "/content/studies_links"
os.environ["EDGEEEG_CACHE"] = "/content/local_cache"
os.environ["EDGEEEG_RESULTS"] = "/content/drive/MyDrive/EdgeEEG_Workspace/runs"

MODALITY = "meg"
NUM_RUNS = 8  # 默认加载 8 个 Run 进行大规模训练；单 Run 快速验证可设为 1
ARCHIVE_PATH = Path(f"/content/drive/MyDrive/EdgeEEG_Workspace/archives/features_{MODALITY}_sub001_runs{NUM_RUNS:02d}.tar")

if ARCHIVE_PATH.exists():
    !{PYTHON} -u /content/code/scripts/cache_archive.py unpack \
        --archive {ARCHIVE_PATH} \
        --cache-dir /content/local_cache
    import json
    vad_manifest = Path("/content/local_cache/manifest_meg_vad.json")
    vad_schema = json.loads(vad_manifest.read_text()).get("cache_schema") if vad_manifest.exists() else None
    if vad_schema != "v3":
        print("Upgrading Stage 1 VAD cache shard to v3 (per-run balanced & standardized)...")
        !{PYTHON} -u -m edge_eeg cache --modality {MODALITY} --stage vad --subject 001 --num-runs {NUM_RUNS} --force-cache --cpu-threads 4
        !{PYTHON} -u /content/code/scripts/cache_archive.py pack --cache-dir /content/local_cache --archive {ARCHIVE_PATH}
else:
    print(f"Archive {ARCHIVE_PATH.name} not found on Drive; building clean v2 cache on local NVMe...")
    !{PYTHON} -u -m edge_eeg cache --modality {MODALITY} --stage vad --subject 001 --num-runs {NUM_RUNS} --cpu-threads 4
    !{PYTHON} -u -m edge_eeg cache --modality {MODALITY} --stage phoneme --subject 001 --num-runs {NUM_RUNS} --cpu-threads 4
    !{PYTHON} -u /content/code/scripts/cache_archive.py pack --cache-dir /content/local_cache --archive {ARCHIVE_PATH}

In [ ]:
# === Cell 4: 第一轮（Stage 1 全量 VAD）— 8 通道 MicroVADNet (INT8 QAT, 目标 85%+ F1-macro) ===
!{PYTHON} -u -m edge_eeg train \
    --modality meg \
    --stage vad \
    --arch student_tiny \
    --channels 8 \
    --subject 001 \
    --num-runs {NUM_RUNS} \
    --epochs 15 \
    --lr 1e-3 \
    --devices 1 \
    --qat \
    --require-cache

In [ ]:
# === Cell 5: 第二轮（Stage 2 探顶 Teacher）— 306 通道 TeacherConformerTCN 探顶并保存 Teacher Logits ===
!{PYTHON} -u -m edge_eeg train \
    --modality meg \
    --stage phoneme \
    --arch teacher_conformer \
    --channels 306 \
    --subject 001 \
    --num-runs {NUM_RUNS} \
    --epochs 12 \
    --lr 1e-3 \
    --batch-size 64 \
    --devices 1 \
    --require-cache

In [ ]:
# === Cell 6: 第三轮（Stage 2 端侧 Student + 蒸馏 + INT8 QAT）— 8 通道 Rep-TCN-EEGNet 导出 model_weights.h ===
!{PYTHON} -u -m edge_eeg train \
    --modality meg \
    --stage phoneme \
    --arch rep_tcn \
    --channels 8 \
    --subject 001 \
    --num-runs {NUM_RUNS} \
    --epochs 16 \
    --lr 2e-3 \
    --devices 1 \
    --distill \
    --qat \
    --require-cache

In [ ]:
# === Cell 7: 汇总打印论文第 3、4、5 章全部最终指标、消融对比与 Stage 3 维特比级联增益 ===
!{PYTHON} -u -m edge_eeg eval \
    --modality meg \
    --stage cascade \
    --subject 001 \
    --num-runs {NUM_RUNS} \
    --verify-codebook